In [11]:
from conch.open_clip_custom import create_model_from_pretrained, tokenize, get_tokenizer
import torch
import os
from PIL import Image
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
# remember 'conda activate conch'


In [ ]:
model_cfg       = 'conch_ViT-B-16'
checkpoint_path = 'checkpoints/conch/pytorch_model.bin'
image_dir       = 'mhist_data/images'
annotations_dir = 'mhist_data/annotations.csv'
device          = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE      = 64

In [4]:
# ── Load model ────────────────────────────────────────────────────────────────
model, preprocess = create_model_from_pretrained(model_cfg, checkpoint_path, device=device)
model = model.eval()
tokenizer = get_tokenizer()

In [5]:
# ── Prompt definitions ────────────────────────────────────────────────────────
# Multiple synonym phrasings per class (descriptions of what's actually visible)
classnames = {
    "HP": [
        "hyperplastic polyp",
        "hyperplastic colorectal polyp",
        "colonic hyperplastic polyp with elongated crypts",
        "benign serrated polyp with superficial luminal serration",
        "hyperplastic polyp with straight crypt bases and goblet cells",
    ],
    "SSA": [
        "sessile serrated adenoma",
        "sessile serrated lesion",
        "sessile serrated adenoma with broad-based dilated crypts",
        "serrated polyp with basal crypt dilation and L-shaped crypts",
        "sessile serrated lesion with horizontal crypt growth and heavy serration",
    ],
}

# Templates — only the descriptive, H&E-explicit and histopathology-grounded ones
templates = [
    "CLASSNAME.",
    "a photomicrograph showing CLASSNAME.",
    "a photomicrograph of CLASSNAME.",
    "a histopathological image showing CLASSNAME.",
    "a histopathological image of CLASSNAME.",
    "a histopathological photograph of CLASSNAME.",
    "a histopathological photograph showing CLASSNAME.",
    "shows CLASSNAME.",
    "presence of CLASSNAME.",
    "CLASSNAME is present.",
    "an H&E stained image of CLASSNAME.",
    "an H&E stained image showing CLASSNAME.",
    "an H&E image showing CLASSNAME.",
    "an H&E image of CLASSNAME.",
    "CLASSNAME, H&E stain.",
    "CLASSNAME, H&E.",
]

In [6]:
@torch.no_grad()
def build_zeroshot_classifier(model, classnames, templates, tokenizer, device):
    """
    For each class, generate all (template × synonym) combinations,
    encode them, average within class, and L2-normalize.
    Returns a tensor of shape [embed_dim, num_classes].
    """
    class_embeddings = []
    class_order = list(classnames.keys())

    for cls in class_order:
        synonyms = classnames[cls]
        # Cartesian product: every template filled with every synonym
        prompts = [t.replace("CLASSNAME", s) for s in synonyms for t in templates]
        print(f"  {cls}: {len(prompts)} prompts (e.g., '{prompts[0]}')")

        tokenized = tokenize(texts=prompts, tokenizer=tokenizer).to(device)
        text_features = model.encode_text(tokenized)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        # average over all prompts → single class embedding, then renormalize
        class_embedding = text_features.mean(dim=0)
        class_embedding = class_embedding / class_embedding.norm()
        class_embeddings.append(class_embedding)

    # stack into [embed_dim, num_classes] for matmul
    weights = torch.stack(class_embeddings, dim=1).to(device)
    return weights, class_order

print("Building zero-shot classifier...")
zeroshot_weights, class_order = build_zeroshot_classifier(
    model, classnames, templates, tokenizer, device
)
print(f"Classifier shape: {zeroshot_weights.shape}")  # [512, 2]

Building zero-shot classifier...
  HP: 80 prompts (e.g., 'hyperplastic polyp.')
  SSA: 80 prompts (e.g., 'sessile serrated adenoma.')
Classifier shape: torch.Size([512, 2])


In [7]:
class MHISTInferenceDataset(torch.utils.data.Dataset):
    def __init__(self, df, image_dir, transform):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.image_dir, row['Image Name'])).convert('RGB')
        return self.transform(image), row['Majority Vote Label']

In [8]:
df = pd.read_csv(annotations_dir)
dataset = MHISTInferenceDataset(df, image_dir, preprocess)
loader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)

In [9]:
@torch.no_grad()
def run_zero_shot(model, loader, zeroshot_weights, class_order, device):
    preds, gts, scores = [], [], []
    for images, labels in tqdm(loader, desc="Zero-shot inference"):
        images = images.to(device)
        image_features = model.encode_image(images)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # logits: [batch, num_classes]
        logits = image_features @ zeroshot_weights
        # CLIP-style temperature scaling (helps if you want softmax probs)
        # logits = 100.0 * logits

        batch_preds = logits.argmax(dim=1).cpu().tolist()
        preds.extend([class_order[p] for p in batch_preds])
        gts.extend(labels)
        scores.append(logits.cpu())

    return preds, gts, torch.cat(scores)

preds, gts, all_scores = run_zero_shot(model, loader, zeroshot_weights, class_order, device)


Zero-shot inference:   0%|          | 0/50 [00:00<?, ?it/s]/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Zero-shot inference: 100%|██████████| 50/50 [15:44<00:00, 18.89s/it]


In [10]:
# ── Metrics ───────────────────────────────────────────────────────────────────
print("\nAccuracy:         ", accuracy_score(gts, preds))
print("Balanced Accuracy:", balanced_accuracy_score(gts, preds))
print("\nClassification report:")
print(classification_report(gts, preds, digits=4))

# Optional: save predictions for further analysis
results_df = pd.DataFrame({
    'image': df['Image Name'],
    'gt': gts,
    'pred': preds,
    'score_HP': all_scores[:, class_order.index('HP')].tolist(),
    'score_SSA': all_scores[:, class_order.index('SSA')].tolist(),
})
results_df.to_csv('mhist_zeroshot_results.csv', index=False)
print("\nSaved per-image predictions to mhist_zeroshot_results.csv")


Accuracy:          0.6903553299492385
Balanced Accuracy: 0.5278782272306787

Classification report:
              precision    recall  f1-score   support

          HP     0.6986    0.9648    0.8104      2162
         SSA     0.5422    0.0909    0.1557       990

    accuracy                         0.6904      3152
   macro avg     0.6204    0.5279    0.4831      3152
weighted avg     0.6495    0.6904    0.6048      3152


Saved per-image predictions to mhist_zeroshot_results.csv


As can be seen in the accuracy of the model it behaves poorly when directly asked to classify an unseen dataset for labels it likely was not directly trained to detect. It's accuracy is only 57%. A model which always predicted HP would get around 68% accuracy. How can we adapt the output CONCH can give to yield better results.